# Predict Customer Churn - Version 2

## Advanced Ensemble Learning with Optuna Hyperparameter Tuning
- **Training Data**: 601K rows (594K competition + 7K IBM)
- **Features**: 54 engineered features
- **Models**: XGBoost + LightGBM + Logistic Regression
- **Ensemble**: Weighted blending + Rank-based blending
- **Submissions**: V3-V6 with different blend weights

## 1. Libraries & Setup

In [ ]:
import numpy as np
import pandas as pd
import os, warnings, pickle
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

import matplotlib.pyplot as plt
import matplotlib.ticker as mtick      # ← added
import seaborn as sns
sns.set_theme(style='whitegrid', font_scale=1.1)

# Models
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier  # ← removed VotingClassifier

# Preprocessing
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import (StratifiedKFold, cross_val_score,
                                     train_test_split)
from sklearn.metrics import roc_auc_score, roc_curve  # ← added roc_curve

# Optuna
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

print('✅ All libraries loaded!')

# File check
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

## 2. Load All Datasets

In [ ]:
# Competition data
train = pd.read_csv('/kaggle/input/competitions/playground-series-s6e3/train.csv')
test  = pd.read_csv('/kaggle/input/competitions/playground-series-s6e3/test.csv')

# Original IBM real data
original = pd.read_csv('/kaggle/input/datasets/thedrzee/customer-churn-in-telecom-sample-dataset-by-ibm/WA_Fn-UseC_-Telco-Customer-Churn.csv')

# Previous submission files for blending
v1 = pd.read_csv('/kaggle/input/datasets/mohankrishnathalla/predict-customer-churn-submission-dataset/V1.csv')  # XGBoost 0.91282
v2 = pd.read_csv('/kaggle/input/datasets/mohankrishnathalla/predict-customer-churn-submission-dataset/V2.csv')  # LR baseline

print(f'Train:    {train.shape}')
print(f'Test:     {test.shape}')
print(f'Original: {original.shape}')
print(f'V1 sub:   {v1.shape}')
print(f'V2 sub:   {v2.shape}')

# Fix original — align Churn column
original['Churn'] = original['Churn'].map({'Yes': 'Yes', 'No': 'No'})

# Add fake id to original
original['id'] = range(900000, 900000 + len(original))

# Combine train + original IBM data
train_combined = pd.concat([train, original], ignore_index=True)
print(f'\nCombined train: {train_combined.shape} ← {len(train)} competition + {len(original)} real IBM')

## 3. Advanced Preprocessing & Feature Engineering

In [ ]:
def preprocess(df, is_train=True):
    df = df.copy()
    
    # Drop ID & customerID
    df.drop(columns=['id', 'customerID'], inplace=True, errors='ignore')
    
    # Fix TotalCharges
    df['TotalCharges'] = pd.to_numeric(
        df['TotalCharges'], errors='coerce'
    ).fillna(0)
    
    # Target encode
    if is_train and 'Churn' in df.columns:
        df['Churn'] = (df['Churn'] == 'Yes').astype(int)
    
    # Binary Yes/No
    binary_cols = ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']
    for col in binary_cols:
        if col in df.columns:
            df[col] = (df[col] == 'Yes').astype(int)
    
    # Gender
    df['gender'] = (df['gender'] == 'Male').astype(int)
    
    # FEATURE ENGINEERING 
    
    # Basic features
    df['AvgMonthlyCharge'] = df['TotalCharges'] / (df['tenure'].clip(lower=1))
    df['ChargeRatio']      = df['MonthlyCharges'] / (df['AvgMonthlyCharge'].clip(lower=0.01))
    df['IsNewCustomer']    = (df['tenure'] <= 12).astype(int)
    df['IsLongTerm']       = (df['tenure'] >= 60).astype(int)
    
    # Total services — fixed (no double count)
    service_cols = ['OnlineSecurity', 'OnlineBackup',
                    'DeviceProtection', 'TechSupport',
                    'StreamingTV', 'StreamingMovies']
    for col in service_cols:
        if col in df.columns and df[col].dtype == object:
            df[col + '_bin'] = (df[col] == 'Yes').astype(int)
    
    # Start with PhoneService (already 0/1)
    df['TotalServices'] = df['PhoneService'].copy()
    for col in ['OnlineSecurity_bin', 'OnlineBackup_bin',
                'DeviceProtection_bin', 'TechSupport_bin',
                'StreamingTV_bin', 'StreamingMovies_bin']:
        if col in df.columns:
            df['TotalServices'] += df[col]
    
    # Charge per service
    df['ChargePerService'] = df['MonthlyCharges'] / (
        df['TotalServices'].clip(lower=1)
    )
    
    # High value customer
    df['IsHighValue'] = (df['MonthlyCharges'] > 70).astype(int)
    
    # No protection flag
    no_protect = []
    for col in ['OnlineSecurity', 'OnlineBackup', 'TechSupport']:
        if col in df.columns:
            no_protect.append((df[col] == 'No').astype(int))
    if no_protect:
        df['NoProtection'] = pd.concat(no_protect, axis=1).sum(axis=1)
    
    # Contract risk score
    if 'Contract' in df.columns:
        contract_risk = {'Month-to-month': 3, 'One year': 2, 'Two year': 1}
        df['ContractRisk'] = df['Contract'].map(contract_risk).fillna(2)
    
    # Interactions
    if 'ContractRisk' in df.columns:
        df['TenureContractRisk']  = df['tenure'] * df['ContractRisk']
        df['ChargeContractRisk']  = df['MonthlyCharges'] * df['ContractRisk']
        df['HighRiskFlag']        = (
            (df['ContractRisk'] == 3) &
            (df['IsNewCustomer'] == 1)
        ).astype(int)
    
    # NEW: Tenure group
    df['TenureGroup'] = pd.cut(
        df['tenure'],
        bins=[0, 12, 24, 36, 48, 60, 72],
        labels=[1, 2, 3, 4, 5, 6]
    ).astype(float).fillna(1)
    
    # NEW: Charge × tenure
    df['ChargeXTenure'] = df['MonthlyCharges'] * df['tenure']
    
    # Drop temp bin cols
    drop_bins = [c for c in df.columns if c.endswith('_bin')]
    df.drop(columns=drop_bins, inplace=True)
    
    # One-Hot Encoding
    ohe_cols = [
        'MultipleLines', 'InternetService',
        'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
        'TechSupport', 'StreamingTV', 'StreamingMovies',
        'Contract', 'PaymentMethod'
    ]
    ohe_cols = [c for c in ohe_cols if c in df.columns]
    df = pd.get_dummies(df, columns=ohe_cols, drop_first=False, dtype=int)
    
    return df


# Apply preprocessing
train_clean = preprocess(train_combined, is_train=True)
test_clean  = preprocess(test, is_train=False)

# Align columns
feature_cols = [c for c in train_clean.columns if c != 'Churn']
for col in feature_cols:
    if col not in test_clean.columns:
        test_clean[col] = 0
test_clean = test_clean[feature_cols]

print(f'Train clean: {train_clean.shape}')
print(f'Test clean:  {test_clean.shape}')
print(f'\nNew features added:')
new_feats = ['TotalServices', 'ChargePerService', 'IsHighValue',
             'NoProtection', 'ContractRisk', 'TenureContractRisk',
             'ChargeContractRisk', 'HighRiskFlag', 'TenureGroup', 'ChargeXTenure']
for f in new_feats:
    print(f'   {f}')

## 4. Setup Training Data & CV

In [ ]:
# Final X, y
y = train_clean['Churn']
X = train_clean.drop(columns=['Churn'])

# Holdout split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# CV strategy — 4 fold (faster than 5, still reliable)
cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=42)

print(f'X_train:  {X_train.shape}')
print(f'X_val:    {X_val.shape}')
print(f'Features: {X.shape[1]}')
print(f'Churn rate train: {y_train.mean():.2%}')
print(f'Churn rate val:   {y_val.mean():.2%}')

# Class imbalance info
churn_ratio      = y_train.mean()
scale_pos_weight = (1 - churn_ratio) / churn_ratio
print(f'\nClass imbalance ratio: {scale_pos_weight:.2f}')
print(f'(Used for XGBoost scale_pos_weight)')

## 5. Optuna Hyperparameter Tuning - LightGBM

In [ ]:
# XGBoost Best Params (pre-loaded)
best_xgb_params = {
    'n_estimators':     967,
    'max_depth':        5,
    'learning_rate':    0.05002318871660704,
    'subsample':        0.771550399322462,
    'colsample_bytree': 0.6620578004729516,
    'min_child_weight': 9,
    'reg_alpha':        5.674678735505923e-06,
    'reg_lambda':       1.1670862425198234,
    'gamma':            0.5592937125390374,
}

print(' XGBoost best params loaded from previous Optuna run!')
print(f'   Previous CV ROC-AUC: 0.91551')
print(f'   n_estimators:        {best_xgb_params["n_estimators"]}')
print(f'   max_depth:           {best_xgb_params["max_depth"]}')
print(f'   learning_rate:       {best_xgb_params["learning_rate"]:.5f}')
print(f'   subsample:           {best_xgb_params["subsample"]:.4f}')
print(f'   colsample_bytree:    {best_xgb_params["colsample_bytree"]:.4f}')

In [ ]:
def objective_lgbm(trial):
    params = {
        'n_estimators':      trial.suggest_int('n_estimators', 300, 2000),
        'max_depth':         trial.suggest_int('max_depth', 3, 10),
        'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'num_leaves':        trial.suggest_int('num_leaves', 20, 150),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
        'subsample':         trial.suggest_float('subsample', 0.6, 1.0),
        'bagging_freq':      1,       
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha':         trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda':        trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
    }
    
    model = LGBMClassifier(
        **params,
        objective='binary',
        metric='auc',
        device='cpu',       
        n_jobs=-1,          
        is_unbalance=True,
        random_state=42,
        verbose=-1
    )
    
    scores = cross_val_score(
        model, X_train, y_train,
        cv=cv, scoring='roc_auc',
        n_jobs=1   
    )
    return scores.mean()


print(' Running Optuna for LightGBM (20 trials)...')
study_lgbm = optuna.create_study(direction='maximize')
study_lgbm.optimize(objective_lgbm, 
                    n_trials=20,          
                    show_progress_bar=True)

print(f'\n LightGBM Best CV ROC-AUC: {study_lgbm.best_value:.5f}')
print(f'Best params: {study_lgbm.best_params}')

## 6. Train Final Models & Validation

In [ ]:
# Final XGBoost 
xgb_final = XGBClassifier(
    **best_xgb_params,              # ← pre-loaded params
    scale_pos_weight=scale_pos_weight,  # ← class imbalance
    objective='binary:logistic',
    eval_metric='auc',
    tree_method='hist',
    device='cuda',
    random_state=42,
    verbosity=0
)
xgb_final.fit(X_train, y_train)
xgb_val_probs = xgb_final.predict_proba(X_val)[:, 1]
xgb_auc = roc_auc_score(y_val, xgb_val_probs)
print(f' XGBoost  Val ROC-AUC: {xgb_auc:.5f}')

# Final LightGBM 
lgbm_final = LGBMClassifier(
    **study_lgbm.best_params,
    objective='binary',
    metric='auc',
    device='cpu',
    n_jobs=-1,
    is_unbalance=True,
    random_state=42,
    verbose=-1
)
lgbm_final.fit(X_train, y_train)
lgbm_val_probs = lgbm_final.predict_proba(X_val)[:, 1]
lgbm_auc = roc_auc_score(y_val, lgbm_val_probs)
print(f' LightGBM Val ROC-AUC: {lgbm_auc:.5f}')

# Logistic Regression 
scale_cols = ['tenure', 'MonthlyCharges', 'TotalCharges',
              'AvgMonthlyCharge', 'ChargeRatio', 'ChargePerService',
              'TenureContractRisk', 'ChargeContractRisk',   # ← added
              'ChargeXTenure']                               # ← added
scale_cols = [c for c in scale_cols if c in X_train.columns]

scaler     = StandardScaler()
X_train_lr = X_train.copy()
X_val_lr   = X_val.copy()
X_train_lr[scale_cols] = scaler.fit_transform(X_train_lr[scale_cols])
X_val_lr[scale_cols]   = scaler.transform(X_val_lr[scale_cols])

lr_final = LogisticRegression(random_state=42, max_iter=1000, C=1.0)
lr_final.fit(X_train_lr, y_train)
lr_val_probs = lr_final.predict_proba(X_val_lr)[:, 1]
lr_auc = roc_auc_score(y_val, lr_val_probs)
print(f' LogReg   Val ROC-AUC: {lr_auc:.5f}')

# Summary
print(f'\n Validation Scores:')
print(f'   XGBoost:  {xgb_auc:.5f}')
print(f'   LightGBM: {lgbm_auc:.5f}')
print(f'   LogReg:   {lr_auc:.5f}')

## 7. Ensemble Weighted Blending

In [ ]:
# Find optimal blend weights using validation set
best_auc    = 0
best_w_xgb  = 0
best_w_lgbm = 0
best_w_lr   = 0

print('🔍 Searching optimal blend weights...')

for w_xgb in np.arange(0.2, 0.8, 0.05):
    for w_lgbm in np.arange(0.1, 0.7, 0.05):
        w_lr = round(1 - w_xgb - w_lgbm, 3)
        if w_lr < 0 or w_lr > 0.4:
            continue
        
        blend = (w_xgb  * xgb_val_probs +
                 w_lgbm * lgbm_val_probs +
                 w_lr   * lr_val_probs)
        
        auc = roc_auc_score(y_val, blend)
        if auc > best_auc:
            best_auc    = auc
            best_w_xgb  = w_xgb
            best_w_lgbm = w_lgbm
            best_w_lr   = w_lr

# Also test individual + pairwise combos
xgb_lgbm_auc = roc_auc_score(
    y_val, 0.5*xgb_val_probs + 0.5*lgbm_val_probs
)

print(f'\n All Scores Compared:')
print(f'   XGBoost only:      {xgb_auc:.5f}')
print(f'   LightGBM only:     {lgbm_auc:.5f}')
print(f'   LogReg only:       {lr_auc:.5f}')
print(f'   XGB + LGBM (50/50):{xgb_lgbm_auc:.5f}')
print(f'   3-Model Ensemble:  {best_auc:.5f} ← best! ')
print(f'\n Best Weights:')
print(f'   XGBoost:  {best_w_xgb:.2f}')
print(f'   LightGBM: {best_w_lgbm:.2f}')
print(f'   LogReg:   {best_w_lr:.2f}')

# Safety check — is ensemble actually better?
individual_best = max(xgb_auc, lgbm_auc, lr_auc)
if best_auc > individual_best:
    print(f'\n Ensemble beats best individual by: '
          f'+{(best_auc - individual_best):.5f}')
else:
    print(f'\n Ensemble does NOT beat best individual model!')
    print(f'   Will use best individual for submission instead')

## 8. ROC Curves

In [ ]:
# Ensemble val probs
ensemble_val = (best_w_xgb  * xgb_val_probs +
                best_w_lgbm * lgbm_val_probs +
                best_w_lr   * lr_val_probs)

models_roc = {
    'Logistic Regression': (lr_val_probs,  lr_auc),
    'XGBoost':             (xgb_val_probs, xgb_auc),
    'LightGBM':            (lgbm_val_probs, lgbm_auc),
    'Ensemble (Best)':     (ensemble_val,  best_auc),
}
colors = ['#4E79A7', '#E15759', '#59A14F', '#F28E2B']

plt.figure(figsize=(9, 7))
for (name, (probs, auc)), color in zip(models_roc.items(), colors):
    fpr, tpr, _ = roc_curve(y_val, probs)
    lw = 3 if 'Ensemble' in name else 2
    plt.plot(fpr, tpr, lw=lw, color=color,
             label=f'{name} (AUC={auc:.4f})')

plt.plot([0,1],[0,1],'k--', lw=1.5, label='Random (AUC=0.500)')
plt.xlim([-0.02, 1.02])
plt.ylim([-0.02, 1.02])
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curve — Competition Models', fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Refit on FULL Data & Generate Predictions

In [ ]:
print('Refitting all models on FULL training data...')
print(f'Total rows: {len(X):,}  (competition + IBM original)')

# XGBoost — full refit 
xgb_full = XGBClassifier(
    **best_xgb_params,                  # ← fixed
    scale_pos_weight=scale_pos_weight,  # ← added
    objective='binary:logistic',
    eval_metric='auc',
    tree_method='hist',
    device='cuda',
    random_state=42,
    verbosity=0
)
xgb_full.fit(X, y)
xgb_test_probs = xgb_full.predict_proba(test_clean)[:, 1]
print(' XGBoost  refit done')

# LightGBM — full refit
lgbm_full = LGBMClassifier(
    **study_lgbm.best_params,
    objective='binary',
    metric='auc',
    device='cpu',
    n_jobs=-1,
    is_unbalance=True,
    random_state=42,
    verbose=-1
)
lgbm_full.fit(X, y)
lgbm_test_probs = lgbm_full.predict_proba(test_clean)[:, 1]
print(' LightGBM refit done')

# Logistic Regression — full refit
X_full_lr    = X.copy()
test_full_lr = test_clean.copy()
X_full_lr[scale_cols]    = scaler.fit_transform(X_full_lr[scale_cols])
test_full_lr[scale_cols] = scaler.transform(test_full_lr[scale_cols])

lr_full = LogisticRegression(random_state=42, max_iter=1000, C=1.0)
lr_full.fit(X_full_lr, y)
lr_test_probs = lr_full.predict_proba(test_full_lr)[:, 1]
print(' LogReg   refit done')

# Sanity check predictions
print(f'\nPrediction ranges:')
print(f'   XGBoost:  min={xgb_test_probs.min():.3f}  max={xgb_test_probs.max():.3f}  mean={xgb_test_probs.mean():.3f}')
print(f'   LightGBM: min={lgbm_test_probs.min():.3f}  max={lgbm_test_probs.max():.3f}  mean={lgbm_test_probs.mean():.3f}')
print(f'   LogReg:   min={lr_test_probs.min():.3f}  max={lr_test_probs.max():.3f}  mean={lr_test_probs.mean():.3f}')

## 10. Final Ensemble & Submission Generation

In [ ]:
from scipy.stats import rankdata

# New ensemble predictions
new_ensemble = (best_w_xgb  * xgb_test_probs +
                best_w_lgbm * lgbm_test_probs +
                best_w_lr   * lr_test_probs)

# Load previous submissions — safe alignment
v1_probs = v1.set_index('id').reindex(test['id'])['Churn'].values
v2_probs = v2.set_index('id').reindex(test['id'])['Churn'].values

# Safety check — no NaN values
assert not np.isnan(v1_probs).any(), 'V1 has NaN after reindex!'
assert not np.isnan(v2_probs).any(), 'V2 has NaN after reindex!'
print(' V1 and V2 alignment verified — no NaN values')

# V3: New Ensemble only 
sub_v3 = pd.DataFrame({'id': test['id'], 'Churn': new_ensemble})
sub_v3.to_csv('submission_v3_ensemble.csv', index=False)
print(' submission_v3_ensemble.csv saved!')

# V4: Blend new ensemble + V1
blend_v4 = np.clip(0.7 * new_ensemble + 0.3 * v1_probs, 0, 1)
sub_v4 = pd.DataFrame({'id': test['id'], 'Churn': blend_v4})
sub_v4.to_csv('submission_v4_blend_v1.csv', index=False)
print(' submission_v4_blend_v1.csv saved!')

# V5: Blend all 3 submissions
blend_v5 = np.clip(
    0.6 * new_ensemble + 0.25 * v1_probs + 0.15 * v2_probs, 0, 1
)
sub_v5 = pd.DataFrame({'id': test['id'], 'Churn': blend_v5})
sub_v5.to_csv('submission_v5_blend_all.csv', index=False)
print(' submission_v5_blend_all.csv saved!')

# V6: Rank-based blend (most stable) 
def rank_blend(*prob_arrays, weights):
    ranked = [rankdata(p) / len(p) for p in prob_arrays]
    return sum(w * r for w, r in zip(weights, ranked))

blend_v6 = rank_blend(
    xgb_test_probs, lgbm_test_probs, lr_test_probs,
    weights=[best_w_xgb, best_w_lgbm, best_w_lr]
)
sub_v6 = pd.DataFrame({'id': test['id'], 'Churn': blend_v6})
sub_v6.to_csv('submission_v6_rank_blend.csv', index=False)
print(' submission_v6_rank_blend.csv saved!')

# Summary
print(f'\n4 submission files ready!')
print(f'V3 (ensemble):    mean={new_ensemble.mean():.4f}')
print(f'V4 (blend V1):    mean={blend_v4.mean():.4f}')
print(f'V5 (blend all):   mean={blend_v5.mean():.4f}')
print(f'V6 (rank blend):  mean={blend_v6.mean():.4f}')

## 11. Distribution Check & Final Summary

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(24, 5))

subs = [
    ('V3 - New Ensemble', new_ensemble, '#4E79A7'),
    ('V4 - Blend + V1',   blend_v4,    '#E15759'),
    ('V5 - Blend All',    blend_v5,    '#59A14F'),
    ('V6 - Rank Blend',   blend_v6,    '#B07AA1'),
]

for ax, (name, probs, color) in zip(axes, subs):
    ax.hist(probs, bins=50, color=color, edgecolor='white', alpha=0.85)
    ax.axvline(0.5, color='black', lw=2, linestyle='--')
    ax.axvline(probs.mean(), color='red', lw=2, 
               linestyle=':', label=f'Mean={probs.mean():.3f}')
    ax.set_title(f'{name}\nMean={probs.mean():.3f}  '
                 f'Std={probs.std():.3f}',
                 fontsize=11, fontweight='bold')
    ax.set_xlabel('Churn Probability')
    ax.set_ylabel('Count')
    ax.legend(fontsize=9)

plt.suptitle('Predicted Probability Distributions — All Submissions',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('=' * 65)
print('COMPETITION NOTEBOOK — FINAL SUMMARY')
print('=' * 65)
print(f'Training rows:    {len(X):,} (competition + IBM)')
print(f'Features:         {X.shape[1]}')
print(f'')
print(f'VALIDATION ROC-AUC:')
print(f'  XGBoost  (Optuna): {xgb_auc:.5f}')
print(f'  LightGBM (Optuna): {lgbm_auc:.5f}')
print(f'  LogReg:            {lr_auc:.5f}')
print(f'  Ensemble (Best):   {best_auc:.5f}')
print(f'')
print(f'  Previous V1 public LB: 0.91282')
print(f'  Expected improvement:  +{(best_auc - 0.91282):.5f}')
print(f'')
print(f'BLEND WEIGHTS:')
print(f'  XGBoost:  {best_w_xgb:.2f}')
print(f'  LightGBM: {best_w_lgbm:.2f}')
print(f'  LogReg:   {best_w_lr:.2f}')
print(f'')
print(f'SUBMISSION FILES:')
print(f'  V3 → New Ensemble only          <- submit first')
print(f'  V4 → Blend with V1 (0.7/0.3)   <- submit second')
print(f'  V5 → Blend all (0.6/0.25/0.15) <- submit third')
print(f'  V6 → Rank blend (most stable)   <- safest private LB')
print(f'')
print(f'SUBMIT STRATEGY:')
print(f'  1. Submit V3 → check public LB')
print(f'  2. Submit V6 → compare with V3')
print(f'  3. Pick where Public LB ≈ CV score')
print(f'  4. Do NOT chase public LB!')
print(f'  5. Save 2 final submissions for deadline')
print('=' * 65)